# 1 — Pipelines and steps

**The concept:** a pipeline is a list of ordinary Python functions. You never build a graph; the
graph is *derived* from how the functions are named.

Two rules do all the work:

- a function's **parameter names** are the variables it consumes
- `outputs=[...]` names the variables it produces

Wiring happens by name. That is the whole model, and everything else in SmartMDAO is a consequence
of it.

In [1]:
from smartmdao import Pipeline

pipeline = Pipeline()

@pipeline.step(outputs=["area"])
def wing_area(span: float, chord: float) -> float:
    return span * chord

@pipeline.step(outputs=["lift"])
def lift_force(area: float, speed: float) -> float:
    return 0.5 * 1.225 * speed**2 * area * 1.2

result = pipeline.run(span=10.0, chord=1.5, speed=50.0)
result

{'span': 10.0, 'chord': 1.5, 'speed': 50.0, 'area': 15.0, 'lift': 27562.5}

`run()` returns a **flat dict of everything** — the inputs you passed plus every variable
produced along the way. Nothing is hidden, and intermediate values stay inspectable.

Notice what we did *not* do: we never said `wing_area` runs before `lift_force`. `lift_force`
consumes `area`, `wing_area` produces it, so the order is implied.

In [2]:
# Registration order does not matter. Declare them backwards and the answer is identical.
backwards = Pipeline()

@backwards.step(outputs=["lift"])
def lift_force_2(area: float, speed: float) -> float:
    return 0.5 * 1.225 * speed**2 * area * 1.2

@backwards.step(outputs=["area"])
def wing_area_2(span: float, chord: float) -> float:
    return span * chord

backwards.run(span=10.0, chord=1.5, speed=50.0)["lift"]

27562.5

## A Step is introspectable without running it

Each registered function becomes a `Step`. A `Step` can answer what it consumes and produces
**from the signature alone** — no execution, no import of your solver, nothing evaluated.

This is the property the entire analysis layer is built on.

In [3]:
from smartmdao import Step

step = pipeline.steps[0]
print("name:     ", step.name)
print("inputs:   ", list(step.get_signature().parameters))
print("outputs:  ", step.resolve_output_names())
print("in types: ", step.resolve_input_types())
print("out types:", step.resolve_output_types())
print("is a Step:", isinstance(step, Step))

name:      wing_area
inputs:    ['span', 'chord']
outputs:   ['area']
in types:  {'span': <class 'float'>, 'chord': <class 'float'>}
out types: {'area': <class 'float'>}
is a Step: True


## Naming outputs

If you omit `outputs=`, the produced variable takes the **function's name**. That is convenient for
a one-output discipline and a trap if you were not expecting it.

In [4]:
implicit = Pipeline()

@implicit.step()                      # no outputs= given
def drag(speed: float) -> float:
    return 0.02 * speed**2

sorted(implicit.run(speed=50.0))

['drag', 'speed']

A function returning a **dataclass** spreads its fields into separate variables, which is how
one discipline produces several coupled outputs without tuple-unpacking by position.

In [5]:
from dataclasses import dataclass

@dataclass
class Aero:
    lift: float
    drag: float

multi = Pipeline()

@multi.step()
def aerodynamics(speed: float, area: float) -> Aero:
    return Aero(lift=0.5 * 1.225 * speed**2 * area, drag=0.02 * speed**2)

multi.run(speed=50.0, area=15.0)

{'speed': 50.0, 'area': 15.0, 'lift': 22968.75, 'drag': 50.0}

## `add()` instead of the decorator

`@pipeline.step` is sugar. `pipeline.add(fn, outputs=[...])` does the same thing and is what you
want when the functions come from somewhere else — a library, a loop, a generated module.

In [6]:
def climb_rate(lift: float, weight: float) -> float:
    return (lift - weight) / weight

assembled = Pipeline()
assembled.add(wing_area, outputs=["area"])
assembled.add(lift_force, outputs=["lift"])
assembled.add(climb_rate, outputs=["climb"])

assembled.run(span=10.0, chord=1.5, speed=50.0, weight=8000.0)["climb"]

2.4453125

## Logging

`configure_logging()` turns on the library's own logging, which is the quickest way to see what the
solver is actually doing step by step.

In [7]:
import logging
from smartmdao import configure_logging

configure_logging(level=logging.INFO)
pipeline.run(span=10.0, chord=1.5, speed=50.0)
logging.getLogger().setLevel(logging.WARNING)   # quiet again for the rest of the notebook
print("done")

22:49:53 | INFO     | smartmdao.core | Starting pipeline execution with 2 steps and inputs: ['span', 'chord', 'speed']


22:49:53 | INFO     | smartmdao.solvers | DAGSolver started.


22:49:53 | INFO     | smartmdao.core | Pipeline execution completed successfully.


done


---

**Next:** [2 — Visualization](02-visualization.ipynb) — before going further, it is worth being able
to *see* the graph these functions just described.